In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib widget

import os, sys
from pathlib import Path
import functools

from extra.components import XrayPulses
import extra_data as ex
from extra_data.components import AGIPD1M
from extra_geom import AGIPD_1MGeometry
from extra_data import by_index
import extra_speckle as esp
from extra_speckle.setup import Setup
from pixel_masker import MaskTool

from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import dask.array as da

import numpy as np
import matplotlib.pyplot as plt
import h5py
from pyFAI.integrator.azimuthal import AzimuthalIntegrator

from analysis_tests.agipd_integrate import _ai

In [2]:
PROPOSAL, RUN_NO = 10400, 423

In [3]:
ai, q, _ = _ai()

In [4]:
run = ex.open_run(run=RUN_NO, proposal=PROPOSAL, data="proc")

In [5]:
exp_setup = Setup(run=run,
                  detector = "AGIPD 1M",
                  configuration = 'SAXS',
                  photon_energy=9.04,
                  sdd = 7531.5962,
                  px = 607.4598195630211,
                  py = 672.076693118667)

/home/andronis/MID-10400/notebooks
detector model: AGIPD 1M
Creating geometry from encoder positions
Encoder positions not found, setting all values to 0
`rep_rate` defined from the pulse pattern
Repetition rate: 2256944.44 Hz (source: AGIPD 1M)
Photon energy provided by the user
photon energy: 9.040 keV
Sample-detector distance (sdd): 7531.596 mm
Using user defined beam center position: (607.46, 672.08)
-------------------



In [6]:
data = AGIPD1M(run.select_trains(by_index[250:251])).get_array('image.data').astype('float32')
mask = AGIPD1M(run.select_trains(by_index[250:251])).get_array('image.mask')

data.data[mask.data>0] = np.nan

data = data.transpose('train', 'pulse', 'module', 'slow_scan', 'fast_scan')
data = data.rename({'train': 'trainId','pulse': 'pulseId', 'module': 'module', 'fast_scan': 'dim_1', 'slow_scan': 'dim_0'}).squeeze()

In [7]:
mean_image = data.mean('pulseId', skipna = True)
mean_image.data[exp_setup.mask] = np.nan